In [1]:
# Connecting to GitHub

setwd("~/stat_app")

In [ ]:
# Installing libraries

install.packages("fixest")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
# Importing libraries

library(haven)
library(dplyr)
library(fixest)

In [ ]:
# Importing data

wave2 <- read_dta("waveII.dta") # midline (cf. pag. 13)
wave3 <- read_dta("waveIII.dta") # endline

Let's remember our econometric equation:

$Y_{icu} = \alpha + \beta_1 I_c + \beta_2 E_c + \beta_3 (I_c \times E_c) + \beta_4' X_{ic} + \epsilon_{icu}$

where $Y_{icu}$ is the outcome for person $i$ in community $c$ and union $u$, $I_c$ is assignment of community $c$ to the incentive program, $E_c$ is assignment of community $c$ to the empowerment program, and $X_{ic}$ is a vector of individual and community controls measured at baseline for strata, age indicators, household size, the presence of an older unmarried sister in the household, school enrollment, mother’s level of education, and whether the community is accessible via public transport (cf. pag. 16).

In [ ]:
# Making the baseline vector, X_ic

controls <- c("older_sister", "bl_still_in_school", "bl_education_mother", "bl_HHsize", "bl_public_transit", "bl_age10", "bl_age11", "bl_age12", "bl_age13",
  "bl_age14", "bl_age15", "bl_age16", "bl_age17", "older_sister_miss", "bl_still_in_school_miss", "bl_education_mother_miss", "bl_HHsize_miss", 
  "bl_public_transit_miss")

In [ ]:
# Preparating the regression

eq <- paste(c("anyemp", "anyoil", "oil_kk", "i(third)", controls), collapse = " + ")

In [ ]:
# Sampling at the midline.

dfw2_all <- wave2 %>% filter(midline == 1, washedout == 0, before_miss == 0, bl_age_reported >= 14, bl_age_reported <= 16)

dfw2_15 <- dfw2_all %>% filter(bl_age_reported == 14)

# Running some regressions

f_ml <- as.formula(paste0("ml_ever_married ~ ", eq, " | unionID"))

regr4 <- feols(f_ml, data = dfw2_all, cluster = ~CLUSTER)
regr5 <- feols(f_ml, data = dfw2_15,  cluster = ~CLUSTER)

In [ ]:
# Sampling at the endline

dfw3_all <- wave3 %>% filter(endline == 1, washedout == 0, before_miss == 0, bl_ever_married == 0, bl_age_reported >= 14, bl_age_reported <= 16)

dfw3_15 <- dfw3_all %>% filter(bl_age_reported == 14)

# Running some regressions

f_u18   <- as.formula(paste0("under_18 ~ ", eq, " | unionID"))
f_u16   <- as.formula(paste0("under_16 ~ ", eq, " | unionID"))
f_mar   <- as.formula(paste0("ever_married ~ ", eq, " | unionID"))
f_mage  <- as.formula(paste0("marriage_age ~ ", eq, " | unionID"))
f_b20   <- as.formula(paste0("ever_birth_20 ~ ", eq, " | unionID"))

regr1 <- feols(f_u18,  data = dfw3_all, cluster = ~CLUSTER)
regr2 <- feols(f_u18,  data = dfw3_15,  cluster = ~CLUSTER)
regr3 <- feols(f_u16,  data = dfw3_15,  cluster = ~CLUSTER)
regr6 <- feols(f_mar,  data = dfw3_all, cluster = ~CLUSTER)
regr7 <- feols(f_mar,  data = dfw3_15,  cluster = ~CLUSTER)
regr8 <- feols(f_mage, data = dfw3_all, cluster = ~CLUSTER)
regr9 <- feols(f_mage, data = dfw3_15,  cluster = ~CLUSTER)
regr10 <- feols(f_b20, data = dfw3_all, cluster = ~CLUSTER)
regr11 <- feols(f_b20, data = dfw3_15,  cluster = ~CLUSTER)

In [ ]:
# Table

etable(regr1, regr2, regr3, regr4, regr5, regr6, regr7, regr8, regr9, regr10, regr11, keep = c("%anyemp","%anyoil","%oil_kk"),
  dict = c(anyemp = "Empowerment", anyoil = "Incentive", oil_kk = "Incen.*Empow."))